# Read-only end-to-end validation

Run after all four ingestion activities and the Silver and Gold notebooks complete. This notebook checks the required metadata, duplicate content versions, SCD Type 2 current rows and validity intervals, Gold resource keys, Raw files and sidecars, and audit/watermark tables. It does not modify project data.

In [ ]:
RAISE_ON_FAILURE = False

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

RESOURCE_TYPES = ["Patient", "Encounter", "Observation", "Condition"]
RAW_ROOT = Path("/lakehouse/default/Files/raw/fhir")
KEY_COLUMNS = {
    "Patient": "patient_id",
    "Encounter": "encounter_id",
    "Observation": "observation_id",
    "Condition": "condition_id"
}

def table_exists(table_name):
    return spark.catalog.tableExists(table_name)

results = []
for resource_type in RESOURCE_TYPES:
    bronze_table = f"bronze.{resource_type}"
    silver_table = f"silver.{resource_type}"
    gold_table = f"gold.{resource_type}"
    required_tables = [bronze_table, silver_table, gold_table]
    missing_tables = [name for name in required_tables if not table_exists(name)]

    row = {
        "resource_type": resource_type,
        "bronze_rows": 0,
        "silver_versions": 0,
        "silver_ids_with_multiple_versions": 0,
        "gold_rows": 0,
        "bronze_missing_required_columns": 0,
        "bronze_required_null_values": 0,
        "silver_missing_required_metadata_columns": 0,
        "silver_required_metadata_null_values": 0,
        "gold_missing_required_metadata_columns": 0,
        "gold_required_metadata_null_values": 0,
        "bronze_duplicate_content_versions": 0,
        "silver_ids_without_exactly_one_current_version": 0,
        "silver_invalid_intervals": 0,
        "gold_missing_ids": 0,
        "gold_duplicate_ids": 0,
        "raw_payload_pages": 0,
        "raw_sidecar_files": 0,
        "successful_audit_rows": 0,
        "watermark_rows": 0,
        "missing_tables": ", ".join(missing_tables),
        "status": "FAIL",
        "reason": ""
    }

    if missing_tables:
        row["reason"] = "One or more layer tables are missing."
        results.append(row)
        continue

    bronze = spark.table(bronze_table)
    silver = spark.table(silver_table)
    gold = spark.table(gold_table)

    required_columns = [
        "resource_id", "resource_json", "content_hash",
        "extraction_timestamp", "api_url_or_params", "saved_timestamp"
    ]
    missing_columns = [name for name in required_columns if name not in bronze.columns]
    row["bronze_missing_required_columns"] = len(missing_columns)
    required_metadata_columns = ["extraction_timestamp", "api_url_or_params"]
    silver_missing_metadata = [
        name for name in required_metadata_columns if name not in silver.columns
    ]
    gold_missing_metadata = [
        name for name in required_metadata_columns if name not in gold.columns
    ]
    row["silver_missing_required_metadata_columns"] = len(silver_missing_metadata)
    row["gold_missing_required_metadata_columns"] = len(gold_missing_metadata)
    row["bronze_rows"] = bronze.count()
    row["silver_versions"] = silver.count()
    row["gold_rows"] = gold.count()

    if not missing_columns:
        for column_name in required_columns:
            row["bronze_required_null_values"] += bronze.filter(
                F.col(column_name).isNull()
            ).count()
        row["bronze_duplicate_content_versions"] = (
            bronze.groupBy("resource_type", "resource_id", "content_hash")
            .count().filter(F.col("count") > 1).count()
        )

    if not silver_missing_metadata:
        for column_name in required_metadata_columns:
            row["silver_required_metadata_null_values"] += silver.filter(
                F.col(column_name).isNull()
            ).count()
    if not gold_missing_metadata:
        for column_name in required_metadata_columns:
            row["gold_required_metadata_null_values"] += gold.filter(
                F.col(column_name).isNull()
            ).count()

    row["silver_ids_without_exactly_one_current_version"] = (
        silver.groupBy("resource_id")
        .agg(F.sum(F.col("is_current").cast("int")).alias("current_count"))
        .filter(F.col("current_count") != 1).count()
    )
    row["silver_invalid_intervals"] = silver.filter(
        F.col("valid_to").isNotNull()
        & (F.col("valid_from").isNull() | (F.col("valid_from") >= F.col("valid_to")))
    ).count()
    row["silver_ids_with_multiple_versions"] = (
        silver.groupBy("resource_id").count().filter(F.col("count") > 1).count()
    )

    key_column = KEY_COLUMNS[resource_type]
    row["gold_missing_ids"] = gold.filter(F.col(key_column).isNull()).count()
    row["gold_duplicate_ids"] = (
        gold.groupBy(key_column).count()
        .filter(F.col(key_column).isNotNull() & (F.col("count") > 1)).count()
    )

    raw_dir = RAW_ROOT / f"resource={resource_type}"
    if raw_dir.exists():
        payload_files = [
            file for file in raw_dir.glob("extraction_date=*/run_id=*/page_*.json")
            if not file.name.endswith(".metadata.json")
        ]
        sidecar_files = list(
            raw_dir.glob("extraction_date=*/run_id=*/page_*.metadata.json")
        )
    else:
        payload_files = []
        sidecar_files = []
    row["raw_payload_pages"] = len(payload_files)
    row["raw_sidecar_files"] = len(sidecar_files)

    if table_exists("dbo.fhir_ingestion_audit"):
        row["successful_audit_rows"] = (
            spark.table("dbo.fhir_ingestion_audit")
            .filter(
                (F.col("resource_type") == resource_type)
                & (F.col("run_status") == "SUCCESS")
            ).count()
        )
    if table_exists("dbo.fhir_ingestion_watermarks"):
        row["watermark_rows"] = (
            spark.table("dbo.fhir_ingestion_watermarks")
            .filter(F.col("resource_type") == resource_type).count()
        )

    failures = []
    if missing_columns:
        failures.append("missing required Bronze columns")
    if row["bronze_required_null_values"] > 0:
        failures.append("null required Bronze metadata")
    if row["silver_missing_required_metadata_columns"] > 0 or row["silver_required_metadata_null_values"] > 0:
        failures.append("missing/null required Silver metadata")
    if row["gold_missing_required_metadata_columns"] > 0 or row["gold_required_metadata_null_values"] > 0:
        failures.append("missing/null required Gold metadata")
    if row["bronze_duplicate_content_versions"] > 0:
        failures.append("duplicate Bronze content versions")
    if row["silver_ids_without_exactly_one_current_version"] > 0:
        failures.append("invalid Silver current-version count")
    if row["silver_invalid_intervals"] > 0:
        failures.append("invalid Silver validity interval")
    if row["gold_missing_ids"] > 0 or row["gold_duplicate_ids"] > 0:
        failures.append("invalid Gold resource IDs")
    if row["raw_payload_pages"] == 0 or row["raw_payload_pages"] != row["raw_sidecar_files"]:
        failures.append("Raw payload/sidecar count mismatch")
    if row["successful_audit_rows"] == 0 or row["watermark_rows"] == 0:
        failures.append("missing successful audit evidence or watermark")

    row["status"] = "PASS" if not failures else "FAIL"
    row["reason"] = "; ".join(failures)
    results.append(row)

print("=== Per-resource validation ===")
display(spark.createDataFrame(results))

control_checks = []
for table_name in ["dbo.fhir_ingestion_audit", "dbo.fhir_ingestion_watermarks"]:
    exists = table_exists(table_name)
    control_checks.append({
        "table": table_name,
        "exists": exists,
        "rows": spark.table(table_name).count() if exists else 0
    })
print("=== Audit and watermark tables ===")
display(spark.createDataFrame(control_checks))

if table_exists("dbo.fhir_ingestion_audit"):
    print("=== Successful runs: retain evidence from separate days ===")
    display(
        spark.table("dbo.fhir_ingestion_audit")
        .filter(F.col("run_status") == "SUCCESS")
        .groupBy("resource_type", "run_id")
        .agg(
            F.count("*").alias("audit_rows"),
            F.min("api_call_timestamp").alias("first_api_call_utc"),
            F.max("saved_timestamp").alias("last_saved_utc")
        )
        .orderBy("resource_type", "run_id")
    )

overall_pass = (
    len(results) == 4
    and all(row["status"] == "PASS" for row in results)
    and all(row["exists"] and row["rows"] > 0 for row in control_checks)
)
print("OVERALL:", "PASS" if overall_pass else "FAIL")
if str(RAISE_ON_FAILURE).strip().lower() in ("true", "1", "yes") and not overall_pass:
    raise RuntimeError("FHIR assessment validation failed. Review the detail outputs.")
